In [ ]:
# Databricks notebook source
# 07_verify_gold_core_pipeline
# Read-only verification for tables produced by the access-drift pipeline.

# Gold Core 파이프라인 읽기 전용 검증

이 노트북은 `access-drift-pipeline` Job 실행 후 생성된 Intermediate, Gold, Eval 테이블을 조회해서 #10 Gold Core 결과를 검증합니다.

검증 범위:

- Intermediate / Gold / Eval 테이블 row count 확인
- Gold Core의 `risk_case_key` 중복 확인
- Risk Card 최소 필수 필드 null 확인
- Gold 결과와 Eval 정답지 key 비교

주의:

- 이 노트북은 read-only 검증 전용입니다.
- 다른 파이프라인 노트북을 다시 호출하지 않습니다.
- seed, bronze, silver, eval, intermediate, gold 데이터를 생성하거나 덮어쓰지 않습니다.

In [ ]:
dbutils.widgets.text("catalog", "access_drift")

CATALOG = dbutils.widgets.get("catalog")

intermediate_table = f"{CATALOG}.intermediate.nhi_residual_access_findings"
gold_table = f"{CATALOG}.gold.gold_core"
eval_table = f"{CATALOG}.eval.ground_truth_case"

print(f"catalog={CATALOG}")
print(f"intermediate_table={intermediate_table}")
print(f"gold_table={gold_table}")
print(f"eval_table={eval_table}")

## 1. 테이블 존재 및 Row Count 확인

Job이 먼저 실행되어 있어야 하며, 이 검증 노트북은 아래 테이블을 읽기만 합니다.

- `intermediate.nhi_residual_access_findings`
- `gold.gold_core`
- `eval.ground_truth_case`

In [ ]:
from pyspark.sql import functions as F

intermediate_df = spark.table(intermediate_table)
gold_df = spark.table(gold_table)
eval_df = spark.table(eval_table)

intermediate_count = intermediate_df.count()
gold_count = gold_df.count()
eval_count = eval_df.count()

print(f"{intermediate_table}: {intermediate_count} rows")
print(f"{gold_table}: {gold_count} rows")
print(f"{eval_table}: {eval_count} rows")

assert intermediate_count >= 1, "expected at least one intermediate finding"
assert gold_count >= 1, "expected at least one gold_core row"
assert eval_count >= 1, "expected at least one eval ground truth row"

## 2. Gold Core 샘플 확인

Risk Card에서 바로 읽어야 하는 최소 결과를 확인합니다.

In [ ]:
display(
    gold_df
    .select(
        "risk_case_key",
        "subject_type",
        "subject_source_principal_id",
        "subject_display_name",
        "asset_source_id",
        "asset_display_name",
        "drift_type",
        "severity",
        "evidence_summary",
        "recommended_action",
        "run_id",
    )
    .orderBy("subject_source_principal_id", "asset_source_id", "risk_case_key")
)

## 3. Key 중복 확인

`risk_case_key`는 Gold Core에서 Risk Card를 식별하는 최소 key입니다.

In [ ]:
duplicate_keys_df = (
    gold_df
    .groupBy("risk_case_key")
    .count()
    .where(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys_df.count()
print(f"duplicate risk_case_key count: {duplicate_key_count}")

display(duplicate_keys_df.orderBy("risk_case_key"))

assert duplicate_key_count == 0, f"duplicate risk_case_key rows: {duplicate_key_count}"

## 4. 필수 필드 Null 확인

Gold Core는 Risk Card 표시와 Silver 근거 추적에 필요한 최소 필드를 null 없이 가져야 합니다.

In [ ]:
required_columns = [
    "risk_case_key",
    "finding_id",
    "subject_type",
    "subject_source_principal_id",
    "subject_display_name",
    "asset_source_id",
    "asset_display_name",
    "drift_type",
    "severity",
    "evidence_summary",
    "recommended_action",
    "owner_edge_id",
    "access_edge_id",
    "credential_id",
]

missing_required_condition = " OR ".join(f"{column} IS NULL" for column in required_columns)
missing_required_df = gold_df.where(F.expr(missing_required_condition))
missing_required_count = missing_required_df.count()

print(f"gold rows with missing required fields: {missing_required_count}")
display(missing_required_df.select(*required_columns).orderBy("risk_case_key"))

assert missing_required_count == 0, f"gold rows with missing required fields: {missing_required_count}"

## 5. Gold-Eval Key 비교

이 비교는 검증 전용입니다. 05/06 탐지 노트북은 eval 테이블을 읽지 않습니다.

비교 key:

```text
gold.subject_type = eval.subject_type
gold.subject_source_principal_id = eval.subject_id
gold.asset_source_id = eval.asset_id
gold.drift_type = eval.drift_type
```

In [ ]:
eval_comparison = (
    gold_df.alias("gold")
    .join(
        eval_df.alias("truth"),
        (F.col("gold.subject_type") == F.col("truth.subject_type")) &
        (F.col("gold.subject_source_principal_id") == F.col("truth.subject_id")) &
        (F.col("gold.asset_source_id") == F.col("truth.asset_id")) &
        (F.col("gold.drift_type") == F.col("truth.drift_type")),
        "left",
    )
    .select(
        F.col("gold.risk_case_key"),
        F.col("gold.subject_source_principal_id"),
        F.col("gold.asset_source_id"),
        F.col("gold.drift_type"),
        F.col("gold.severity"),
        F.col("truth.case_id"),
        F.col("truth.is_planted_risk"),
        F.col("truth.expected_severity"),
        F.col("truth.label_reason"),
    )
    .orderBy("subject_source_principal_id", "asset_source_id", "risk_case_key")
)

matched_truth_rows = eval_comparison.where(F.col("case_id").isNotNull()).count()
unmatched_gold_rows = gold_count - matched_truth_rows

print(f"gold rows matched to eval truth: {matched_truth_rows} / {gold_count}")
print(f"gold rows without eval match: {unmatched_gold_rows}")

display(eval_comparison)

assert matched_truth_rows >= 1, "expected at least one gold row to match eval truth"

## 6. 검증 결과

위 assertion이 모두 통과하면 #10 Gold Core 산출물에 대한 read-only 검증이 완료된 것입니다.

In [ ]:
print("Gold Core read-only verification passed")